# 7. ONNX Export

Loads the trained MLP model from R2 and exports it to ONNX format, then saves the `.onnx` file back to `keywordsdb/models/mlp/` on R2.

ONNX is useful for running inference outside PyTorch (e.g. ONNX Runtime, TensorRT, edge devices).

Steps:
1. Add `service_account.json` secret with a service account with access to R2.
2. Run training notebook (5_training) first so `model.pt` exists.
3. No GPU needed — CPU is fine for the export.


In [ ]:
%pip install onnxscript onnxruntime --quiet

In [ ]:
import io
import json

import numpy as np
import onnxruntime as ort
import s3fs  # ty:ignore[unresolved-import]
import torch
import torch.nn as nn
from kaggle_secrets import UserSecretsClient  # ty:ignore[unresolved-import]

user_secrets = UserSecretsClient()
r2_creds = json.loads(user_secrets.get_secret("service_account.json"))
print("Authenticated with R2 access key:", r2_creds.get("access_key_id"))

R2_MODELS = "keywordsdb/models"
MODEL_PT_PATH = f"{R2_MODELS}/mlp/model.pt"
MODEL_ONNX_PATH = f"{R2_MODELS}/mlp/model.onnx"

EMBEDDING_DIM = 384
INPUT_DIM = EMBEDDING_DIM + 5  # 384 embedding + 5 text features

INTENT_LABELS = ["informational", "commercial", "transactional", "navigational"]
ENTITY_LABELS = ["person", "event", "location", "company", "product", "other"]

# Output order used by the ONNX model (must match OnnxWrapper below)
ONNX_OUTPUT_NAMES = [
    "intent",  # logits (N, 4)
    "entity_type",  # logits (N, 6)
    "is_adult",  # raw regression (N, 1)
    "saas_potential",  # raw regression (N, 1)
    "is_question",  # logits (N, 1)
    "is_comparison",  # logits (N, 1)
    "is_branded",  # logits (N, 1)
]

FS = s3fs.S3FileSystem(
    key=r2_creds["access_key_id"], secret=r2_creds["secret_access_key"], endpoint_url=r2_creds["endpoint_url"]
)

In [ ]:
task_configs = [
    ("intent", "multiclass", len(INTENT_LABELS)),
    ("entity_type", "multiclass", len(ENTITY_LABELS)),
    ("is_adult", "regression", 1),
    ("saas_potential", "regression", 1),
    ("is_question", "binary", 1),
    ("is_comparison", "binary", 1),
    ("is_branded", "binary", 1),
]


class MultiTaskMLP(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(INPUT_DIM, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.1),
        )
        self.heads = nn.ModuleDict({task_key: nn.Linear(256, n_out) for task_key, _, n_out in task_configs})

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        h = self.backbone(x)
        return {task_key: self.heads[task_key](h) for task_key in self.heads}


class OnnxWrapper(nn.Module):
    """Wraps MultiTaskMLP to return a flat tuple of tensors.

    ONNX does not support dict outputs, so we return tensors in the fixed
    order defined by ONNX_OUTPUT_NAMES.
    """

    def __init__(self, model: MultiTaskMLP) -> None:
        super().__init__()
        self.model = model

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, ...]:
        out = self.model(x)
        return tuple(out[name] for name in ONNX_OUTPUT_NAMES)


# Load weights
model = MultiTaskMLP()
with FS.open(MODEL_PT_PATH, "rb") as f:
    state_dict = torch.load(io.BytesIO(f.read()), map_location="cpu")
model.load_state_dict(state_dict)
model.eval()
print("Loaded model.pt")

In [ ]:
wrapper = OnnxWrapper(model)
wrapper.eval()

# Dummy input — batch of 2 samples
dummy_input = torch.randn(2, INPUT_DIM)

# Export using the dynamo-based exporter (default since PyTorch 2.9).
# dynamic_shapes uses torch.export.Dim; "x" matches OnnxWrapper.forward(self, x).
batch_dim = torch.export.Dim("batch_size")
onnx_program = torch.onnx.export(
    wrapper, (dummy_input,), input_names=["input"], output_names=ONNX_OUTPUT_NAMES, dynamic_shapes={"x": {0: batch_dim}}
)
assert onnx_program is not None

onnx_buf = io.BytesIO()
onnx_program.save(onnx_buf)  # ty:ignore[invalid-argument-type]
onnx_buf.seek(0)
onnx_bytes = onnx_buf.read()
print(f"ONNX model size: {len(onnx_bytes) / 1024:.1f} KB")

In [ ]:
# Verify: run the same dummy input through both models and compare outputs
with torch.no_grad():
    pt_outputs = wrapper(dummy_input)

sess = ort.InferenceSession(onnx_bytes, providers=["CPUExecutionProvider"])
ort_outputs = sess.run(None, {"input": dummy_input.numpy()})

max_diff = max(float(np.abs(pt.numpy() - ort).max()) for pt, ort in zip(pt_outputs, ort_outputs))
print(f"Max absolute difference between PyTorch and ONNX Runtime outputs: {max_diff:.2e}")
assert max_diff < 1e-4, f"Outputs differ too much: {max_diff}"
print("Verification passed.")

In [ ]:
with FS.open(MODEL_ONNX_PATH, "wb") as f:
    f.write(onnx_bytes)
print(f"Saved: {MODEL_ONNX_PATH}")